# 04 Feature Engineering

This notebook illustrates how to derive new features from the raw data.
Use it to experiment with ratios, buckets and other transformations
before moving them into `src/features/build_features.py`.

In [1]:
from pathlib import Path
import zipfile

project_root = Path('/workspaces/Brokerflow_ai')
zip_path = project_root / 'data-science-nigeria-challenge-1-loan-default-prediction20250307-26022-im3qg9.zip'
print('zip exists:', zip_path.exists())
if zip_path.exists():
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        print('zip entries:', len(names))
        for name in names[:50]:
            print(name)

print('\nCSV files already present:')
for path in sorted(project_root.rglob('*.csv')):
    print(path.relative_to(project_root))

zip exists: True
zip entries: 8
manifest-155273bc0bf3f6c96da245fb19b9dc9320250307-26022-7yk61d.json
trainperf.csv
testprevloans.zip
trainprevloans.zip
SampleSubmission.csv
testdemographics.csv
testperf.csv
traindemographics.csv

CSV files already present:
data/synthetic/applications.csv
data/synthetic/booking_notes_example.csv
data/synthetic/documents.csv
data/synthetic/reviews.csv


In [3]:
import csv
from io import BytesIO, TextIOWrapper

with zipfile.ZipFile(zip_path) as zf:
    for csv_name in ['traindemographics.csv', 'trainperf.csv', 'testdemographics.csv', 'testperf.csv', 'SampleSubmission.csv']:
        with zf.open(csv_name) as f:
            reader = csv.reader(TextIOWrapper(f, encoding='utf-8'))
            rows = list(reader)
            print(f'\n{csv_name}: rows={len(rows) - 1}, cols={len(rows[0]) if rows else 0}')
            if rows:
                print('columns:', rows[0])
                for sample in rows[1:3]:
                    print(sample)

    for nested_name in ['trainprevloans.zip', 'testprevloans.zip']:
        with zf.open(nested_name) as nested_file:
            nested_bytes = BytesIO(nested_file.read())
            with zipfile.ZipFile(nested_bytes) as nested_zip:
                print(f'\n{nested_name}: entries={nested_zip.namelist()}')
                for inner_name in nested_zip.namelist():
                    with nested_zip.open(inner_name) as inner_file:
                        reader = csv.reader(TextIOWrapper(inner_file, encoding='utf-8'))
                        rows = list(reader)
                        print(f'{inner_name}: rows={len(rows) - 1}, cols={len(rows[0]) if rows else 0}')
                        if rows:
                            print('columns:', rows[0])
                            for sample in rows[1:3]:
                                print(sample)


traindemographics.csv: rows=4346, cols=9
columns: ['customerid', 'birthdate', 'bank_account_type', 'longitude_gps', 'latitude_gps', 'bank_name_clients', 'bank_branch_clients', 'employment_status_clients', 'level_of_education_clients']
['8a858e135cb22031015cbafc76964ebd', '1973-10-10 00:00:00.000000', 'Savings', '3.319219', '6.5286039', 'GT Bank', '', '', '']
['8a858e275c7ea5ec015c82482d7c3996', '1986-01-21 00:00:00.000000', 'Savings', '3.3255983', '7.1194033', 'Sterling Bank', '', 'Permanent', '']

trainperf.csv: rows=4368, cols=10
columns: ['customerid', 'systemloanid', 'loannumber', 'approveddate', 'creationdate', 'loanamount', 'totaldue', 'termdays', 'referredby', 'good_bad_flag']
['8a2a81a74ce8c05d014cfb32a0da1049', '301994762', '12', '2017-07-25 08:22:56.000000', '2017-07-25 07:22:47.000000', '30000.0000', '34500.0000', '30', '', 'Good']
['8a85886e54beabf90154c0a29ae757c0', '301965204', '2', '2017-07-05 17:04:41.000000', '2017-07-05 16:04:18.000000', '15000.0000', '17250.0000', '

In [4]:
from collections import Counter

with zipfile.ZipFile(zip_path) as zf:
    def load_csv_dicts(name):
        with zf.open(name) as f:
            return list(csv.DictReader(TextIOWrapper(f, encoding='utf-8')))

    train_demo = load_csv_dicts('traindemographics.csv')
    train_perf = load_csv_dicts('trainperf.csv')
    test_demo = load_csv_dicts('testdemographics.csv')
    test_perf = load_csv_dicts('testperf.csv')

    train_demo_ids = {row['customerid'] for row in train_demo}
    train_perf_ids = {row['customerid'] for row in train_perf}
    test_demo_ids = {row['customerid'] for row in test_demo}
    test_perf_ids = {row['customerid'] for row in test_perf}

    print('train demo ids:', len(train_demo_ids))
    print('train perf ids:', len(train_perf_ids))
    print('train perf without demographics:', len(train_perf_ids - train_demo_ids))
    print('train demographics without perf:', len(train_demo_ids - train_perf_ids))
    print('sample missing in demographics:', list(sorted(train_perf_ids - train_demo_ids))[:5])

    print('\ntest demo ids:', len(test_demo_ids))
    print('test perf ids:', len(test_perf_ids))
    print('test perf without demographics:', len(test_perf_ids - test_demo_ids))
    print('test demographics without perf:', len(test_demo_ids - test_perf_ids))
    print('sample missing in demographics:', list(sorted(test_perf_ids - test_demo_ids))[:5])

    malformed_test_dates = [row['approveddate'] for row in test_perf if ':' in row['approveddate'] and '-' not in row['approveddate']]
    print('\nmalformed approveddate in testperf:', len(malformed_test_dates))
    print('sample malformed values:', malformed_test_dates[:5])

train demo ids: 4334
train perf ids: 4368
train perf without demographics: 1099
train demographics without perf: 1065
sample missing in demographics: ['8a1edbf14734127f0147356fdb1b1eb2', '8a2ac4745091002b0150a144bcbe58b7', '8a390a2150ad97330150aebdd8ef7456', '8a399f284741b63b0147496a04db1eb4', '8a5cfa8345d40fb80145dc328532397a']

test demo ids: 1484
test perf ids: 1450
test perf without demographics: 1065
test demographics without perf: 1099
sample missing in demographics: ['8a28afc7474813a40147639ec637156b', '8a3735d5518aba7301518ac34413010d', '8a76e7d443e6e97c0143ed099d102b1d', '8a818823525dceef01525deda2480384', '8a818926522ea5ef01523aff15c37482']

malformed approveddate in testperf: 1450
sample malformed values: ['40:48.0', '43:40.0', '15:11.0', '00:54.0', '04:33.0']


In [5]:
print('train_perf in test_demo:', len(train_perf_ids & test_demo_ids))
print('test_perf in train_demo:', len(test_perf_ids & train_demo_ids))
print('train_demo in test_perf:', len(train_demo_ids & test_perf_ids))
print('test_demo in train_perf:', len(test_demo_ids & train_perf_ids))

train_perf in test_demo: 1099
test_perf in train_demo: 1065
train_demo in test_perf: 1065
test_demo in train_perf: 1099


## Feature Engineering sur les données Zindi enrichies

Les cellules suivantes chargent les tables enrichies via `raw_competition.py` et construisent des features dérivées (ratios, buckets, historique).


In [3]:
%pip install -q pandas numpy
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import numpy as np
import pandas as pd
from src.data import load_enriched_raw_competition_tables

ROOT = pathlib.Path('..').resolve()
ZIP_PATH = ROOT / 'data-science-nigeria-challenge-1-loan-default-prediction20250307-26022-im3qg9.zip'
OUT_DIR = ROOT / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_raw, test_raw = load_enriched_raw_competition_tables(str(ZIP_PATH))

def add_raw_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['loanamount_safe'] = out['loanamount'].replace({0: 1e-6})
    out['due_vs_loan_ratio'] = out['totaldue'] / out['loanamount_safe']
    out['log_loanamount'] = np.log1p(out['loanamount'].clip(lower=0))
    out['log_totaldue'] = np.log1p(out['totaldue'].clip(lower=0))

    if 'firstduedate' in out.columns and 'approveddate' in out.columns:
        out['days_to_due'] = (
            pd.to_datetime(out['firstduedate'], errors='coerce')
            - pd.to_datetime(out['approveddate'], errors='coerce')
        ).dt.days
    else:
        out['days_to_due'] = np.nan

    out['is_savings_account'] = (out.get('bank_account_type', pd.Series(index=out.index, dtype='object')).fillna('') == 'Savings').astype(int)

    if 'birthdate' in out.columns and 'approveddate' in out.columns:
        out['customer_age_approx'] = (
            pd.to_datetime(out['approveddate'], errors='coerce').dt.year
            - pd.to_datetime(out['birthdate'], errors='coerce').dt.year
        )
    else:
        out['customer_age_approx'] = np.nan

    return out

train_feat = add_raw_features(train_raw)
test_feat = add_raw_features(test_raw)

print(f"Train raw  : {train_raw.shape} -> features: {train_feat.shape}")
print(f"Test raw   : {test_raw.shape} -> features: {test_feat.shape}")
new_cols = [c for c in train_feat.columns if c not in train_raw.columns]
print(f"Nouvelles colonnes ({len(new_cols)}) : {new_cols}")

train_feat.to_csv(OUT_DIR / 'train_features.csv', index=False)
test_feat.to_csv(OUT_DIR / 'test_features.csv', index=False)
print(f"Features sauvegardées dans: {OUT_DIR}")


Note: you may need to restart the kernel to use updated packages.


/workspaces/Brokerflow_ai/src/data/raw_competition.py:123: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  prepared["approveddate"] = pd.to_datetime(prepared["approveddate"], errors="coerce")
/workspaces/Brokerflow_ai/src/data/raw_competition.py:124: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  prepared["creationdate"] = pd.to_datetime(prepared["creationdate"], errors="coerce")


Train raw  : (4368, 26) -> features: (4368, 33)
Test raw   : (1450, 24) -> features: (1450, 31)
Nouvelles colonnes (7) : ['loanamount_safe', 'due_vs_loan_ratio', 'log_loanamount', 'log_totaldue', 'days_to_due', 'is_savings_account', 'customer_age_approx']
Features sauvegardées dans: /workspaces/Brokerflow_ai/data/processed
